# 03 — Nested evaluation on both views

Every variant is evaluated separately on each view with exp_1's nesting: temperature and decoder settings are chosen on the six inner validation matches (pair models) of that view, the held-out match is scored once. Final configs (for BAR-ATH) are selected on the 7 out-of-fold predictions.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp2_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_2')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
import exp2_common as C
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp2_protocol as P
import exp1_common as C1

PREREG verified 154a10f7cda6821e | frozen 2026-11-08T13:06:55+00:00


In [2]:
cache = {}
for variant in P.VARIANTS:
    for view in C.VIEWS:
        t0 = time.time()
        result = P.evaluate_variant(variant, view, cache)
        C1.write_json(C.RESULTS / f'variant_{variant}__{view}.json', result)
        frame = pd.DataFrame(result['rows'])
        print(f'== {variant:15s} {view:12s} mean spellF1 {frame.spell_f1.mean():.4f}  max|count err| {frame.count_error_ratio.abs().max():.3f}  ({time.time() - t0:.0f}s)')
        print('   final config:', {k: round(v, 3) for k, v in result['final_config'].items()})

== exp1_ref        original     mean spellF1 0.6556  max|count err| 0.342  (77s)
   final config: {'temperature': 1.303, 'lambda_team': 0.5, 'lambda_dead': 0.5, 'gamma': 0.0}


== exp1_ref        target_like  mean spellF1 0.6479  max|count err| 0.342  (73s)
   final config: {'temperature': 1.295, 'lambda_team': 0.5, 'lambda_dead': 1.0, 'gamma': 0.0}


== exp2_aug        original     mean spellF1 0.6567  max|count err| 0.354  (73s)
   final config: {'temperature': 1.299, 'lambda_team': 1.0, 'lambda_dead': 2.0, 'gamma': 0.0}


== exp2_aug        target_like  mean spellF1 0.6566  max|count err| 0.354  (73s)
   final config: {'temperature': 1.282, 'lambda_team': 1.0, 'lambda_dead': 4.0, 'gamma': 0.0}


== exp2_aug_const  original     mean spellF1 0.6567  max|count err| 0.354  (17s)
   final config: {'temperature': 1.299, 'lambda_team': 1.0, 'lambda_dead': 2.0, 'gamma': 0.0}


== exp2_aug_const  target_like  mean spellF1 0.6566  max|count err| 0.354  (17s)
   final config: {'temperature': 1.282, 'lambda_team': 1.0, 'lambda_dead': 4.0, 'gamma': 0.0}


... (4 outputs left out)

== exp2_aug_s2     original     mean spellF1 0.6564  max|count err| 0.335  (74s)
   final config: {'temperature': 1.279, 'lambda_team': 0.5, 'lambda_dead': 8.0, 'gamma': 0.0}


== exp2_aug_s2     target_like  mean spellF1 0.6557  max|count err| 0.337  (75s)
   final config: {'temperature': 1.263, 'lambda_team': 1.0, 'lambda_dead': 8.0, 'gamma': 0.0}


== exp2_aug_ens3   original     mean spellF1 0.6662  max|count err| 0.360  (78s)
   final config: {'temperature': 1.163, 'lambda_team': 0.5, 'lambda_dead': 1.0, 'gamma': 0.0}


== exp2_aug_ens3   target_like  mean spellF1 0.6641  max|count err| 0.366  (77s)
   final config: {'temperature': 1.153, 'lambda_team': 0.5, 'lambda_dead': 4.0, 'gamma': 0.0}


## Per-match detail (held-out matches)

In [3]:
keep = ['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'count_error_ratio']
for variant in ('exp1_ref', 'exp2_aug', 'exp2_aug_hsmm', 'exp2_aug_ens3'):
    for view in C.VIEWS:
        frame = pd.DataFrame(C1.read_json(C.RESULTS / f'variant_{variant}__{view}.json')['rows'])
        extra = [c for c in ('gamma', 'weight', 'lambda_team', 'lambda_dead', 'temperature') if c in frame]
        print(f'== {variant} / {view}'); print(frame[keep + extra].round(3).to_string(index=False))

== exp1_ref / original
 match  spell_f1  macro_f1  n_true  n_pred  count_error_ratio  gamma  lambda_team  lambda_dead  temperature
J03WMX     0.617     0.920     511     336             -0.342    0.0          1.0         16.0        1.286
J03WN1     0.644     0.904     500     356             -0.288    0.0          1.0          8.0        1.331
J03WOH     0.675     0.920     402     318             -0.209    0.0          0.5          1.0        1.285
J03WOY     0.666     0.920     456     372             -0.184    0.0          1.0          2.0        1.285
J03WPY     0.683     0.927     421     343             -0.185    0.0          1.0          4.0        1.294
J03WQQ     0.688     0.921     401     383             -0.045    0.0          1.0          8.0        1.284
J03WR9     0.616     0.912     370     285             -0.230    0.0          1.0          2.0        1.274
== exp1_ref / target_like
 match  spell_f1  macro_f1  n_true  n_pred  count_error_ratio  gamma  lambda_team  lamb

## Diagnostic: argmax (no decoder)

In [4]:
argmax = pd.concat([pd.DataFrame(P.evaluate_argmax(source, view, cache))
                    for source in ('exp1_ref', 'exp2_aug') for view in C.VIEWS], ignore_index=True)
argmax.to_csv(C.RESULTS / 'argmax_rows.csv', index=False)
print(argmax.groupby(['variant', 'view'])[['spell_f1', 'macro_f1', 'count_error_ratio']].mean().round(4).to_string())

                             spell_f1  macro_f1  count_error_ratio
variant         view                                              
exp1_ref_argmax original       0.6240    0.9176            -0.0390
                target_like    0.6159    0.9123            -0.0285
exp2_aug_argmax original       0.6282    0.9174            -0.0548
                target_like    0.6266    0.9166            -0.0529
